# Notebook Header & Import

In [3]:
# EEEM068: Industrial Waste Classification
# Evaluation Notebook (Jupyter version of test.py)

import os
import json
import argparse

from sklearn import metrics
import yaml
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# reuse model builder from train.py
from train import get_model, get_transforms, save_confusion_matrix, load_class_mapping

# Markdown: Evaluation Overview

In [4]:
## 🧪 Evaluation Pipeline Overview

#This notebook:
#- Loads a saved experiment folder (config + checkpoint)
#- Rebuilds the model
#- Loads the test dataset
#- Runs full evaluation
#- Computes:
#  - accuracy
#  - macro F1
#  - weighted F1
#  - macro precision
#  - macro recall
#  - AUC (one-vs-rest)
#  - mAP
#  - per-class metrics
#- Saves:
#  - test_metrics.json
#  - confusion matrix
#- Optionally runs GradCAM (extra credit)

# Evaluation Function

In [5]:
@torch.no_grad()
def evaluate(model, loader, device, n_classes):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

    return (
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_probs)
    )

# Markdown: Metric Computation


In [6]:
## 📊 Metric Computation

#This cell computes:
#- accuracy
#- macro/weighted F1
#- macro precision/recall
#- AUC (one-vs-rest)
#- mAP
#- per-class classification report

# Metric Function


In [7]:
def compute_metrics(labels, preds, probs, n_classes, class_names=None):
    metrics_dict = {
        "accuracy": float(np.mean(labels == preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(labels, preds, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels, preds, average="macro", zero_division=0)),
    }

    # AUC
    try:
        one_hot = np.eye(n_classes)[labels]
        metrics_dict["auc"] = float(
            roc_auc_score(one_hot, probs, average="macro", multi_class="ovr")
        )
    except Exception as e:
        metrics_dict["auc"] = None
        print(f"[Warning] AUC could not be computed: {e}")

    # mAP
    try:
        one_hot = np.eye(n_classes)[labels]
        metrics_dict["mAP"] = float(
            average_precision_score(one_hot, probs, average="macro")
        )
    except Exception as e:
        metrics_dict["mAP"] = None
        print(f"[Warning] mAP could not be computed: {e}")

    # Per-class report
    metrics_dict["per_class_report"] = classification_report(
        labels, preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    return metrics_dict

# Markdown: Save Test Metrics


In [8]:
## 💾 Saving Test Metrics

#This saves:
#- `test_metrics.json`
#- prints a summary to the notebook

#  Save Metrics Function


In [9]:
def save_test_metrics(metrics, output_dir):
    path = os.path.join(output_dir, "test_metrics.json")
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"[Metrics] Test results saved to {path}\n")

    print("="*40)
    print("  TEST RESULTS")
    print("="*40)
    for k, v in metrics.items():
        if k == "per_class_report":
            continue
        print(f"{k:<20} {v}")
    print("="*40)

# Cell 8 — Markdown: Optional GradCAM

In [10]:
## 🔥 GradCAM (Extra Credit)

# This is optional and requires:

In [11]:
pip install grad-cam

  Using cached grad_cam-1.5.5-py3-none-any.whl
  Using cached ttach-0.0.3-py3-none-any.whl.metadata (5.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Using cached ttach-0.0.3-py3-none-any.whl (9.8 kB)

   ---------- ----------------------------- 1/4 [tqdm]
   ---------- ----------------------------- 1/4 [tqdm]
   -------------------- ------------------- 2/4 [opencv-python]
   -------------------- ------------------- 2/4 [opencv-python]
   -------------------- ------------------- 2/4 [opencv-python]
   -------------------- ------------------- 2/4 [opencv-python]
   ------------------------------ --------- 3/4 [grad-cam]
   ------------------------------ --------- 3/4 [grad-cam]
   ------------------------------ --------- 3/4 [grad-cam]
   -------------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# GradCAM Stub

In [12]:
def run_gradcam(model, loader, device, output_dir, n_samples=8):
    try:
        from pytorch_grad_cam import GradCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image

        print("[GradCAM] TODO: implement full GradCAM visualisation here")

    except ImportError:
        print("[GradCAM] Install pytorch-grad-cam: pip install grad-cam")

# Markdown: Main Evaluation Cell

In [13]:
## 🚀 Run Evaluation

#This cell:
#- loads config.yaml from a run folder  
#- loads best_model.pth  
#- rebuilds model  
#- loads test dataset  
#- runs evaluation  
#- saves metrics + confusion matrix  

# Main Evaluation Logic

In [14]:
def run_evaluation(run_folder, enable_gradcam=False):
    # Load config
    config_path = os.path.join(run_folder, "config.yaml")
    with open(config_path) as f:
        config = yaml.safe_load(f)

    print(f"Evaluating run: {config['run']['name']}")
    print(f"Model: {config['run']['model']}")

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    # Model
    model = get_model(config).to(device)
    ckpt_path = os.path.join(run_folder, "best_model.pth")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"[Model] Loaded checkpoint from {ckpt_path}")

    # Test loader
    test_tf = get_transforms(config, train=False)
    test_dataset = datasets.ImageFolder(
        root=os.path.join(config["dataset"]["data_root"], "test"),
        transform=test_tf
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config["evaluation"]["batch_size"],
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Class names
    mapping = load_class_mapping(config)
    class_names = [mapping["label_to_class"][str(i)]
                   for i in range(config["dataset"]["n_classes"])]

    # Evaluate
    preds, labels, probs = evaluate(model, test_loader, device, len(class_names))

    # Metrics
    metrics = compute_metrics(labels, preds, probs, len(class_names), class_names)
    save_test_metrics(metrics, run_folder)

    # Confusion matrix
    if config["output"]["save_confusion_matrix"]:
        save_confusion_matrix(labels, preds, class_names, run_folder)

    # GradCAM
    if enable_gradcam or config["output"].get("save_gradcam", False):
        run_gradcam(model, test_loader, device, run_folder)

    print(f"[Done] All test outputs saved to: {run_folder}")

# Example Usage


In [16]:
# Example: evaluate a run
run_evaluation("experiments/results/resnet50/smoke_test")

Evaluating run: smoke_test
Model: resnet50
[Device] cuda
[Model] Loading resnet50 (pretrained=False)
[Model] Params — total: 23,565,404 | trainable: 23,565,404
[Model] Loaded checkpoint from experiments/results/resnet50/smoke_test\best_model.pth
[Metrics] Test results saved to experiments/results/resnet50/smoke_test\test_metrics.json

  TEST RESULTS
accuracy             0.218568665377176
f1_macro             0.17038897217135598
f1_weighted          0.18962459103969762
precision_macro      0.255810183214227
recall_macro         0.21834600205034924
auc                  0.8154694795878712
mAP                  0.21261587791434308
[Plot] Confusion matrix saved to experiments/results/resnet50/smoke_test\confusion_matrix.png
[Done] All test outputs saved to: experiments/results/resnet50/smoke_test
